In [27]:
import pandas as pd
import sys
import os
module_dir = os.path.abspath('utils') 
sys.path.append(module_dir)
import csv_file_upload as cu

In [12]:
def separate_likert_from_text(dataframe: pd.DataFrame, likert_max_unique: int = 10) -> dict[list]:
    likert_columns = []
    text_columns   = []
    other_columns  = []

    for col_name in dataframe.columns:
        column_data = dataframe[col_name].dropna() #extracts column as a seriees

        #check if column is empty
        if len(column_data) == 0:
            other_columns.append(col_name)
            continue

        #Check 1: if values are numeric
        is_numeric = pd.api.types.is_numeric_dtype(column_data)

        #Check2: count number of unique values
        unique_count = column_data.unique()

        #Check 4: Contains alphabetic chars?
        has_letters = column_data.astype(str).str.contains('[a-zA-Z]').any()

        #----Decision Logic------
        if is_numeric and not has_letters:
            likert_columns.append(col_name)
        elif not is_numeric and has_letters:
            text_columns.append(col_name)
        else:
            other_columns.append(col_name)

    return {'likert': likert_columns, 'text':   text_columns, 'other':  other_columns}

In [13]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
#Input:  likert columns dataframe (pd.DataFrame) — likert columns from the survey
# Output: a 2D numpy array where each row is one respondent's normalized Likert scores

def normalize_likert(likert_df) -> np.array: 

    # MinMaxScaler is a transformer from scikit-learn.
    # It rescales each column independently so that the minimum value in that column
    # becomes 0.0 and the maximum value becomes 1.0. All values in between are
    # proportionally scaled. 
    scaler = MinMaxScaler()

    # .fit_transform() does two things in one call:
    # 1. .fit() — the scaler learns the min and max of each column from the data
    # 2. .transform() — it applies the scaling formula to convert values to 0–1
    # The result is a 2D numpy array with shape (num_respondents, num_likert_columns)
    normalized_likert = scaler.fit_transform(likert_df)
    print(f"Normalized Likert array shape: {normalized_likert.shape}")
    return normalized_likert  

In [14]:
# This function converts a list of text responses into a 2D array of embeddings.
# Each text response becomes a row of 384 numbers that captures its semantic meaning.
#
# Input:  dataframe (pd.DataFrame) — your full survey DataFrame
#         text_columns (list of str) — the names of the open-ended text columns
# Output: a 2D numpy array of shape (num_respondents, 384)

from sentence_transformers import SentenceTransformer

def embed_text_responses(text_df) -> np.array:
    # SentenceTransformer loads a pretrained embedding model.
    # 'all-MiniLM-L6-v2' is a widely used lightweight model. It:
    #   - Produces 384-dimensional embeddings
    #   - Is fast enough to run on a standard laptop CPU
    #   - Performs well on English survey text
    # The model files are downloaded automatically on first use (~80MB).
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # We concatenate all open-ended text columns for each respondent into one string.
    # This way, each respondent is represented by all their text together, not separately.
    #
    # .fillna("") replaces any missing (NaN) values with an empty string,
    # because the embedding model cannot process Python's NaN (Not a Number) values.
    #
    # .agg(" ".join, axis=1) applies a function across each row (axis=1 means row-wise).
    # For each row, it joins the values of all selected text columns with a space between them.
    combined_text = text_df.fillna("").agg(" ".join, axis=1)

    # Convert the pandas Series to a plain Python list.
    # The .encode() method below expects a list of strings.
    text_list = combined_text.tolist()

    # .encode() runs each string through the embedding model and returns
    # a 2D numpy array. Each row is one respondent's embedding vector.
    # show_progress_bar=True prints a progress indicator, useful for larger datasets.
    embeddings = model.encode(text_list, show_progress_bar=True)

    print(f"Text embedding array shape: {embeddings.shape}")
    # Expected output: (num_respondents, 384)

    return embeddings

In [15]:
def combine_features(normalized_likert, text_embeddings):
    combined = np.hstack([normalized_likert, text_embeddings])
    print(f"Combined feature array shape: {combined.shape}")
    return combined

In [16]:
# This function reduces the dimensionality of the combined feature array using PCA.
# It is optional but recommended for large feature vectors.
#
# Input:  combined_features (2D numpy array) — from Stage 4
#         n_components (int) — how many dimensions to reduce to (default: 50)
#                              A good starting point; you can increase or decrease.
# Output: a 2D numpy array with shape (num_respondents, n_components)

from sklearn.decomposition import PCA

def reduce_dimensions(combined_features, n_components=50):

    # Cap n_components at the number of features available.
    # You cannot reduce to more dimensions than you started with.
    n_components = min(n_components, combined_features.shape[1])

    # PCA() initializes the PCA transformer.
    # n_components tells it how many dimensions to keep.
    # random_state=42 makes the result reproducible — PCA has a random initialization step,
    # and setting this ensures you get the same result every time you run it.
    pca = PCA(n_components=n_components, random_state=42)

    # .fit_transform() learns the principal components from the data
    # and immediately applies the transformation.
    reduced = pca.fit_transform(combined_features)

    # explained_variance_ratio_ is an array that tells you what fraction of the
    # total variation in the data each component captures.
    # Summing them tells you how much information is retained overall.
    variance_retained = sum(pca.explained_variance_ratio_) * 100
    print(f"Reduced to {n_components} dimensions, retaining {variance_retained:.1f}% of variance.")

    return reduced

In [17]:
# This function automatically selects the best number of clusters
# by running K-Means for each candidate k and computing the silhouette score.
# It returns the best k and the labeled array — no chart needed.
#
# Input:
#   features    (2D numpy array) — the PCA-reduced combined feature array
#   min_k       (int)            — smallest k to test (default: 2)
#                                  Cannot be 1 — silhouette score is undefined for 1 cluster
#   max_k       (int)            — largest k to test (default: 10)
#                                  Set this to a reasonable upper bound for your dataset size
# Output:
#   best_k          (int)              — the k with the highest silhouette score
#   best_labels     (1D numpy array)   — cluster assignment for each respondent
#   scores_summary  (dict)             — all scores, so you can inspect them if curious

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

def auto_cluster(features, min_k=2, max_k=10):

    # This dictionary will store the silhouette score for each k we test.
    # Key = k (number of clusters), Value = silhouette score
    scores = {}

    # This dictionary stores the cluster labels produced by each k,
    # so we do not have to re-run K-Means after finding the best k.
    all_labels = {}

    for k in range(min_k, max_k + 1):

        # Run K-Means with this value of k.
        # n_init=10 runs the algorithm 10 times with different random starts
        # and keeps the best result — this reduces the chance of a poor solution.
        # random_state=42 makes results reproducible.
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)

        # .fit_predict() trains the model and returns cluster labels in one step.
        # labels is a 1D array like [0, 2, 1, 0, 3, ...] — one integer per respondent.
        labels = kmeans.fit_predict(features)

        # silhouette_score() computes the average silhouette score across all points.
        # It needs:
        #   features — the actual data points (so it can measure distances)
        #   labels   — the cluster assignment for each point
        # A higher score means clusters are more distinct and well-separated.
        score = silhouette_score(features, labels)

        # Store both the score and the labels for this k
        scores[k] = score
        all_labels[k] = labels

        print(f"  k={k:2d}  →  silhouette score: {score:.4f}")

    # Find the k with the highest silhouette score.
    # max() with a key function finds the dictionary key whose value is largest.
    # scores.get is passed as the key function — it looks up the score for each k.
    best_k = max(scores, key=scores.get)

    print(f"\n✓ Best k = {best_k}  (silhouette score: {scores[best_k]:.4f})")

    # Retrieve the labels that were already computed for the best k.
    best_labels = all_labels[best_k]

    # Build a summary dict so you can inspect all scores if you want to
    scores_summary = {
        "best_k": best_k,
        "best_score": scores[best_k],
        "all_scores": scores
    }

    return best_k, best_labels, scores_summary

In [18]:
# This function adds the cluster assignment back to the original DataFrame
# so you can inspect which respondents are in each cluster and read their responses.
#
# Input:  dataframe (pd.DataFrame) — your original survey data
#         cluster_labels (1D numpy array) — from Stage 7
#         text_columns (list of str) — columns to display for inspection
# Output: the original DataFrame with a new 'cluster' column added;
#         also prints sample responses per cluster
def attach_and_inspect_clusters(dataframe, cluster_labels, text_columns):
    # .copy() creates a full copy of the DataFrame so we do not accidentally
    # modify the original data in place
    labeled_df = dataframe.copy()

    # Add the cluster assignments as a new column called 'cluster'
    labeled_df['cluster'] = cluster_labels

    return labeled_df

In [21]:
# Main pipeline function that runs all stages end-to-end.
#
# Input:  file_path (str) — path to your CSV
#         likert_columns (list of str) — column names for Likert questions
#         text_columns (list of str) — column names for open-ended questions
#         n_clusters (int or None) — number of clusters to use;
#         if None, the elbow chart is shown first
# Output: the labeled DataFrame with a 'cluster' column

#using the read_csv_file():

def run_clustering_pipeline(n_clusters=None):
    # Stage 1: Load
    survey_df = cu.load_survey_data()

    # Stage 2: Likert and Text Columns Extraction
    likert_cols, text_cols, other_cols = separate_likert_from_text(survey_df).values()
    likert_df = survey_df[likert_cols]
    text_df = survey_df[text_cols]
    
    # Stage 2: Normalize Likert
    normalized_likert = normalize_likert(survey_df, likert_cols)
    
    # Stage 3: Embed text
    text_embeddings = embed_text_responses(survey_df, text_cols)

    # Stage 4: Combine
    combined_features = combine_features(normalized_likert, text_embeddings)

    # Stage 5: Reduce dimensions
    reduced_features = reduce_dimensions(combined_features, n_components=50)

    # Stage 6 & 7: Handle cluster count
    if n_clusters is None:
        # Show the elbow chart so you can pick k manually
        plot_elbow_curve(reduced_features, min_k=2, max_k=10)
        print("\nInspect the chart above and re-run with your chosen n_clusters value.")
        return None

    # Stage 7: Cluster
    cluster_labels = run_clustering(reduced_features, n_clusters)

    # Stage 8: Inspect
    labeled_df = attach_and_inspect_clusters(survey_df, cluster_labels, text_columns)

    return [labeled_df, likert_df, text_df]

In [20]:
# --- EXAMPLE USAGE ---
# Replace these with your actual column names after inspecting the CSV.

# First run: show elbow chart to pick k
result = run_clustering_pipeline(
    file=file_csv,
    n_clusters=None        # <-- leave as None to see the elbow chart first
)

# Second run: once you've picked k (say, 4) from the chart
result = run_clustering_pipeline(
    file=file_csv,
    n_clusters=3           # <-- replace with your chosen number
)

NameError: name 'file_csv' is not defined

In [ ]:
def label_cluster_with_llm(labeled_df, cluster_id, avg_ratings, n_sample):
    responses = labeled_df.loc[labeled_df['cluster']== cluster_id].to_numpy()
    sample_responses = responses[:n_sample]
    

In [ ]:
display(main_df.columns)

In [ ]:
labeled_df = attach_and_inspect_clusters(main_df, cluster_labels, main_df.columns)

In [ ]:
cluster_id = 1

In [ ]:
responses = labeled_df.loc[labeled_df['cluster']== cluster_id].to_numpy()

In [ ]:
avg_ratings = labeled_df.groupby('cluster')[likert_cols].mean().round(2)

In [ ]:
sample_responses = responses[:]
print(sample_responses)

In [ ]:
formatted_responses = "\n".join(
        f"{i+1}. {text}" for i, text in enumerate(sample_responses)
    )
    # Format the average ratings into a readable string.
    # .items() returns key-value pairs from the dictionary.
    # We round to 1 decimal place for readability.
formatted_ratings = ", ".join(
    f"{key}: {round(val, 1)}/5" for key, val in avg_ratings.items()
)

In [ ]:
prompt = f"""You are analyzing post-program evaluation survey data.
Below are {len(sample_responses)} responses from Cluster {cluster_id}, 
along with that cluster's average Likert ratings.

AVERAGE RATINGS:
{formatted_ratings}

SAMPLE RESPONSES:
{formatted_responses}

Based on these responses and ratings, provide a cluster label in JSON format.
Return ONLY a JSON object with no extra text, no markdown, no code fences.
Use this exact structure:
{{
  "label": "A short 2-4 word descriptive name for this cluster. Description should include the clusters common theme, main concerns and praises, and other opinions that is relevant to the scope of the program.",
  "summary": "One sentence describing who these respondents and their sentiments and opinions",
  "actionable insights: "Descriptive list of insights and potential direction or next steps for the organizers. Include insights that would benefit the program as a whole. Reference or cite specific respondents for these insights"
  "themes": ["theme 1", "theme 2", "theme 3"]
}}"""

In [ ]:
from nlp.llm_client import call_llm_with_retry as call_llm

In [ ]:
labeled_df.columns.to_numpy()

In [ ]:
print(call_llm(f"Judge base on the survey questions: {text_df.columns.to_numpy()}.",prompt))

In [ ]:
display(labeled_df.loc[labeled_df['cluster']== cluster_id])